# SMA Module 2 — Working with the data

IPO lists, daily prices, and a simple RSI rule. Run cells from top to bottom.


## 0) Imports


In [ ]:
import os
# keep caches on E: — C: fills up fast on this machine
os.environ.setdefault("MPLCONFIGDIR", r"E:\IT_SPACES\AI\.cache\matplotlib")
os.environ.setdefault("XDG_CACHE_HOME", r"E:\IT_SPACES\AI\.cache")
os.environ.setdefault("TEMP", r"E:\IT_SPACES\AI\.cache\tmp")
os.environ.setdefault("TMP", r"E:\IT_SPACES\AI\.cache\tmp")

from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import yfinance as yf

# the timezone cache was landing on C:, so I pointed it at E:
yf.set_tz_cache_location(r"E:\IT_SPACES\AI\.cache\py-yfinance")


## Question 1: Withdrawn IPOs by Company Type

Which company type has the highest total withdrawn IPO value, in $ millions?

Source: [Recently filed IPOs](https://www.iposcoop.com/ipos-recently-filed/).


In [ ]:
url = "https://www.iposcoop.com/ipos-recently-filed/"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}
response = requests.get(url, headers=headers, timeout=60)
response.raise_for_status()
# pandas tried to open the HTML text as a file path, so I wrapped it in StringIO
filed = pd.read_html(StringIO(response.text))[0]
withdrawn = filed.loc[filed["Expected To Trade"] == "Withdrawn"].copy()
# the prompt said 32 withdrawn names; this page had 34 when I ran it
print(len(withdrawn))


In [ ]:
def company_type(name):
    # first match wins. "Technology" is not "Technologies", and "Corp" is checked before "Inc"
    rules = [
        ("Technologies", "Technologies"),
        ("Acquisition Corp", "Acquisition Corp"),
        ("Acquisition Corporation", "Acquisition Corp"),
        ("Corp", "Acquisition Corp"),
        ("Inc", "Inc."),
        ("Incorporated", "Inc."),
        ("Group", "Group"),
        ("Ltd", "Limited"),
        ("Limited", "Limited"),
        ("Holdings", "Holdings"),
        ("Holding", "Holdings"),
    ]
    text = "" if pd.isna(name) else str(name)
    for needle, label in rules:
        if needle in text:
            return label
    return "Other"


def to_num(series):
    cleaned = (
        series.astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
        .replace({"-": np.nan, "": np.nan, "nan": np.nan})
    )
    return pd.to_numeric(cleaned, errors="coerce")


withdrawn["Company Type"] = withdrawn["Company"].map(company_type)
low = to_num(withdrawn["Price Low"])
high = to_num(withdrawn["Price High"])
withdrawn["Avg_price"] = (low + high) / 2
shares = to_num(withdrawn["Shares (millions)"])
est = to_num(withdrawn["Est $ Vol (millions)"])
product = shares * withdrawn["Avg_price"]
# a "-" price made the product empty, so those rows use Est $ Vol instead
withdrawn["Shares_offered_value"] = product.where(product.notna(), est)
totals = (
    withdrawn.groupby("Company Type")["Shares_offered_value"]
    .sum()
    .sort_values(ascending=False)
)
print(totals)
print(totals.idxmax(), round(float(totals.max()), 3))


On my run the largest class was **Acquisition Corp**, **499.985** ($ millions). Cutting the list at 11 Sep 2026 left that same total.


## Question 2: Median Sharpe for 2025 IPOs (first 8 months)

Median Sharpe on 11 Sep 2026 for IPOs offered before 1 Sep 2025.

Source: [2025 IPO pricings](https://www.iposcoop.com/2025-pricings/).


In [ ]:
url = "https://www.iposcoop.com/2025-pricings/"
response = requests.get(url, headers=headers, timeout=60)
response.raise_for_status()
ipos = pd.read_html(StringIO(response.text))[0]
ipos["Offer Date"] = pd.to_datetime(ipos["Offer Date"], format="%m/%d/%Y")
ret = pd.to_numeric(ipos["Return"].astype(str).str.replace("%", "", regex=False), errors="coerce")
# the prompt said 148; I got 146 after dropping a 0% return and a blank symbol
keep = ipos.loc[(ipos["Offer Date"] < "2025-09-01") & (ret != 0)].copy()
keep = keep.loc[keep["Symbol"].notna() & (keep["Symbol"].astype(str).str.strip() != "")]
tickers = keep["Symbol"].astype(str).str.strip().unique().tolist()
print(len(keep), len(tickers))


In [ ]:
# one big download locked yfinance's timezone database, so I pulled 20 names at a time
frames = []
for i in range(0, len(tickers), 20):
    batch = tickers[i : i + 20]
    raw = yf.download(
        batch,
        start="2025-01-01",
        end="2026-09-12",
        interval="1d",
        auto_adjust=False,
        threads=True,
        progress=False,
        timeout=20,
    )
    if raw is None or len(raw) == 0:
        continue
    # recent yfinance returns a MultiIndex, so Close was not a flat column
    if isinstance(raw.columns, pd.MultiIndex):
        close = raw["Close"].copy()
    else:
        close = raw[["Close"]].copy()
        close.columns = batch[:1]
    if isinstance(close, pd.Series):
        close = close.to_frame(name=batch[0])
    got = [c for c in close.columns if close[c].notna().any()]
    if got:
        frames.append(close[got])

close = pd.concat(frames, axis=1)
close = close.loc[:, ~close.columns.duplicated()]
stocks_df = close.stack(future_stack=True).rename("Close").reset_index()
stocks_df.columns = ["Date", "Ticker", "Close"]
# the index was timezone-aware and would not match 2026-09-11, so I stripped it
stocks_df["Date"] = pd.to_datetime(stocks_df["Date"]).dt.tz_localize(None)
stocks_df = stocks_df.dropna(subset=["Close"]).sort_values(["Ticker", "Date"])
# delisted symbols 404'd; the prompt said about 134, I kept 132 with a Close
print(stocks_df["Ticker"].nunique())


In [ ]:
# shift(252) without a groupby mixed the previous ticker into the next one
stocks_df["growth_252d"] = stocks_df["Close"] / stocks_df.groupby("Ticker")["Close"].shift(252)
stocks_df["volatility"] = (
    stocks_df.groupby("Ticker")["Close"].transform(lambda s: s.rolling(30).std()) * np.sqrt(252)
)
stocks_df["Sharpe"] = (stocks_df["growth_252d"] - 0.05) / stocks_df["volatility"]

day = stocks_df.loc[stocks_df["Date"] == "2026-09-11"].copy()
# a flat 30-day close made volatility 0 and Sharpe infinite, so I dropped those
day = day.replace([np.inf, -np.inf], np.nan)
print(day[["growth_252d", "volatility", "Sharpe"]].describe())
print("median Sharpe", float(day["Sharpe"].median()))
print("mean growth", float(day["growth_252d"].mean()), "median growth", float(day["growth_252d"].median()))


On my run the median Sharpe on 11 Sep 2026 was **0.049**. Mean `growth_252d` was about 1.05 and the median was about 0.59, so a few big winners pull the average up.


## Question 3: Fixed months holding strategy

Which holding period from 1 to 12 months has the highest median growth from the IPO close?

Same `stocks_df` as Question 2. One month is 21 trading days.


In [ ]:
for month in range(1, 13):
    stocks_df[f"future_growth_{month}_m"] = (
        stocks_df.groupby("Ticker")["Close"].shift(-21 * month) / stocks_df["Close"]
    )

# keep only each ticker's first row, so the ratio is versus the IPO close
entry = stocks_df.loc[stocks_df.groupby("Ticker")["Date"].idxmin()].copy()
growth_cols = [f"future_growth_{month}_m" for month in range(1, 13)]
# one tiny first close blew up the mean (into the thousands); the median stays usable
print(entry[growth_cols].describe())
medians = entry[growth_cols].median().sort_values(ascending=False)
print(medians)
print(medians.index[0], round(float(medians.iloc[0]), 3))


On my run the best hold is **1 month**, median growth **0.935**. The median falls as the hold gets longer, and it stays under 1, so the typical new IPO in this list is down from the first close.


## Question 4: RSI strategy

Total profit, in $ thousands, from investing $1000 every time RSI is under 30, from 2000-01-01 to 2025-06-01.


In [ ]:
import gdown

file_id = "1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-"
dest = Path("data.parquet")
# gdown was missing from this venv, and this version has no fuzzy= argument
if not dest.exists():
    gdown.download(f"https://drive.google.com/uc?id={file_id}", str(dest), quiet=False)
rsi = pd.read_parquet(dest, engine="pyarrow")
print(rsi.shape)
print(rsi.columns.tolist())


In [ ]:
signal = rsi.loc[
    (pd.to_datetime(rsi["Date"]) >= "2000-01-01")
    & (pd.to_datetime(rsi["Date"]) <= "2025-06-01")
    & (rsi["rsi"] < 30)
].copy()
# the column is lowercase "rsi"; filtering on "RSI" matched nothing
print(len(signal))
net_income = 1000 * (signal["growth_future_30d"] - 1).sum()
print(round(float(net_income), 1), round(float(net_income) / 1000, 1))


On my run that was **5,206** trades and **65.8** ($ thousands). Average 30-day growth was 1.26% and the win rate was 55.1%, same ballpark as the prompt.


## Question 5: A more profitable IPO rule

I would stop buying every new listing. In Question 3 the median growth was below 1 at every horizon and got worse the longer I held, while the mean was inflated by one tiny first close. I would drop Acquisition Corp / blank-check names (they were the whole top of the withdrawn-value list and often have no business yet), drop symbols Yahoo cannot price, and keep a name only if the first-month close is above the IPO close. Among those, I would rank by the Question 2 Sharpe and take the top half instead of holding the full list for a year.
